# CTU-13 Dataset EDA

Exploratory data analysis for all 13 CTU-13 botnet scenarios.
Requires flow files under `data/raw/` (set `DATA_DIR` below to override).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from gbd_ctu.data.ctu13_loader import CTU13Loader, dataset_statistics_table, scenario_statistics

DATA_DIR = Path('data/raw')   # override if needed
OUTPUT_DIR = Path('results/eda')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data dir : {DATA_DIR.resolve()}')
print(f'Output dir: {OUTPUT_DIR.resolve()}')

In [ ]:
# Cell 2 — Build statistics table
loader = CTU13Loader(data_root=DATA_DIR)
stats_df = dataset_statistics_table(loader)

display_cols = [
    'scenario_id', 'family', 'total_flows',
    'botnet_flows', 'botnet_frac',
    'normal_flows', 'background_flows',
    'unique_src_ips', 'unique_dst_ips',
    'botnet_host_ips', 'unique_dst_ports',
    'mean_duration', 'max_duration',
]
display_cols = [c for c in display_cols if c in stats_df.columns]
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
stats_df[display_cols]

In [ ]:
# Cell 3 — Per-scenario class imbalance bar chart
fig, ax = plt.subplots(figsize=(12, 4))
x = stats_df['scenario_id'].astype(str)
ax.bar(x, stats_df['botnet_frac'] * 100, label='Botnet %', color='#d62728')
ax.bar(x, stats_df['normal_frac'] * 100,
       bottom=stats_df['botnet_frac'] * 100, label='Normal %', color='#1f77b4')
ax.bar(x, stats_df['background_frac'] * 100,
       bottom=(stats_df['botnet_frac'] + stats_df['normal_frac']) * 100,
       label='Background %', color='#aec7e8')
ax.set_xlabel('Scenario ID')
ax.set_ylabel('Flow fraction (%)')
ax.set_title('CTU-13 Class Composition per Scenario')
ax.legend(loc='upper right')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'class_imbalance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 4 — Flow duration distributions (violin plot)
all_frames = loader.load_all()   # loads all scenarios
duration_data = {
    sid: frame['duration'].clip(0, frame['duration'].quantile(0.99)).values
    for sid, frame in all_frames.items()
}
labels = [str(k) for k in sorted(duration_data)]
data   = [duration_data[k] for k in sorted(duration_data)]

fig, ax = plt.subplots(figsize=(14, 5))
parts = ax.violinplot(data, showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor('#1f77b4')
    pc.set_alpha(0.6)
ax.set_xticks(range(1, len(labels) + 1))
ax.set_xticklabels(labels)
ax.set_xlabel('Scenario ID')
ax.set_ylabel('Flow duration (s, 99th pct. clip)')
ax.set_title('Flow Duration Distribution per CTU-13 Scenario')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'flow_duration_violin.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 5 — Botnet vs benign degree CDF (using unique_dst_count per src_addr)
# Aggregate all scenarios for a combined view
all_rows = pd.concat(all_frames.values(), ignore_index=True)
degree = (
    all_rows.groupby(['src_addr', 'label_binary'])['dst_addr']
    .nunique()
    .reset_index()
    .rename(columns={'dst_addr': 'out_degree'})
)
botnet_deg  = degree[degree['label_binary'] == 1]['out_degree'].values
benign_deg  = degree[degree['label_binary'] == 0]['out_degree'].values

fig, ax = plt.subplots(figsize=(8, 5))
for vals, label, color in [
    (botnet_deg, 'Botnet', '#d62728'),
    (benign_deg, 'Benign', '#1f77b4'),
]:
    sorted_vals = np.sort(vals)
    cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
    ax.plot(sorted_vals, cdf, label=label, color=color)
ax.set_xscale('log')
ax.set_xlabel('Out-degree (unique destination IPs per src)')
ax.set_ylabel('CDF')
ax.set_title('Botnet vs Benign Out-Degree CDF (all scenarios)')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'degree_cdf.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Cell 6 — Export LaTeX table
latex_cols = [
    'scenario_id', 'family', 'total_flows',
    'botnet_flows', 'botnet_frac',
    'unique_src_ips', 'botnet_host_ips',
    'mean_duration',
]
latex_cols = [c for c in latex_cols if c in stats_df.columns]
tex = stats_df[latex_cols].to_latex(
    index=False,
    float_format='%.4f',
    na_rep='—',
    caption='CTU-13 per-scenario dataset statistics.',
    label='tab:dataset_stats',
)
tex_path = OUTPUT_DIR / 'dataset_stats.tex'
tex_path.write_text(tex, encoding='utf-8')
print(f'LaTeX table written to {tex_path}')
print(tex)